# Phase 4 — Data Science Check
Train a RandomForest model with MLflow tracking.
Uses 1% sample to fit in local memory.

In [ ]:
import os
os.chdir(r"D:\Retail Demand Forecasting")

In [ ]:
from pyspark.sql import SparkSession
from retail_demand_forecasting.nodes.data_engineering import unpivot_sales
from retail_demand_forecasting.nodes.feature_engineering import create_features
from retail_demand_forecasting.nodes.data_science import train_model

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("phase4_data_science")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .config("spark.local.dir", r"D:\spark-temp")
    .getOrCreate()
)
print(f"Spark {spark.version} ready.")

## 1. Prepare features

In [ ]:
PROJECT_ROOT = r"D:\Retail Demand Forecasting"

sales_train_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/sales_train_validation.csv"))
calendar_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/calendar.csv"))
sell_prices_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(os.path.join(PROJECT_ROOT, "data/01_raw/sell_prices.csv"))

melted_df = unpivot_sales(sales_train_raw, calendar_raw, sell_prices_raw)
melted_df = melted_df.sample(fraction=0.01, seed=42)

params = {"lag_days": [7, 28], "rolling_window_days": [7, 28]}
featured_df = create_features(melted_df, params)

# Convert to pandas for sklearn
feature_pd = featured_df.toPandas()
print(f"Feature matrix: {feature_pd.shape[0]:,} rows x {feature_pd.shape[1]} cols")

## 2. Train model

In [ ]:
train_params = {
    "model_params": {"max_depth": 5, "num_trees": 50},
    "target_col": "sales",
    "test_size": 0.2,
    "random_state": 42,
}

metrics = train_model(feature_pd, train_params)

## 3. Metrics

In [ ]:
print(f"MAE:  {metrics['mae']:.4f}")
print(f"RMSE: {metrics['rmse']:.4f}")
print(f"MAPE: {metrics['mape']:.2f}%")
print(f"Features: {metrics['features']}")

In [ ]:
spark.stop()
print("Phase 4 complete.")